In [14]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# 將專案 root 加入 python path（讓 src/ 可以 import）
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)
print("✓ Imports ready")

Project root added: C:\Users\USER\Desktop\2026_japan
✓ Imports ready


In [1]:
import joblib
model = joblib.load("../models/lgbm_sapporo_2.pkl")

features = [
    "weekday", "t", "x", "y", "is_weekend",
    "lag_1", "lag_7", "rolling_3", "rolling_7"
]

print("✓ Model loaded")

✓ Model loaded


In [2]:
import pandas as pd

df = pd.read_parquet("../data/processed/sapporo_density.parquet")
is_unknown = (df["x"]==999) & (df["y"]==999)
df = df[~is_unknown].copy() 

df["date"] = pd.to_datetime("2023-01-01") + pd.to_timedelta(df["d"], unit="D")
df["weekday"] = df["date"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

In [3]:
df = df.sort_values(["x","y","t","d"])

df["lag_1"] = df.groupby(["x","y","t"])["count"].shift(1)
df["lag_7"] = df.groupby(["x","y","t"])["count"].shift(7)

df["rolling_3"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(3).mean())
)

df["rolling_7"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df_feat = df.dropna().copy()
print("after feature rows:", len(df_feat))

after feature rows: 4881569


In [4]:
import numpy as np
import pandas as pd
import lightgbm as lgb

features = ["weekday","t","x","y","is_weekend","lag_1","lag_7","rolling_3","rolling_7"]
infer_df = df_feat.copy()

infer_df["score"] = model.predict(infer_df[features])
infer_df["score"] = infer_df["score"].clip(lower=0)

df_pred = infer_df[["d","t","x","y","score"]].copy()

In [5]:
import re
from dataclasses import dataclass

SLOTS_PER_DAY = 48
SLOT_MIN = 24*60 // SLOTS_PER_DAY  # 30

WEEKDAY_MAP = {
    "週一":1, "星期一":1, "禮拜一":1,
    "週二":2, "星期二":2, "禮拜二":2,
    "週三":3, "星期三":3, "禮拜三":3,
    "週四":4, "星期四":4, "禮拜四":4,
    "週五":5, "星期五":5, "禮拜五":5,
    "週六":6, "星期六":6, "禮拜六":6,
    "週日":0, "星期日":0, "星期天":0, "禮拜日":0,
}

@dataclass
class CrowdQuery:
    city: str
    weekday: int | None      # 0=Mon..6=Sun (or your own convention; just be consistent)
    hhmm: str | None         # "20:00"
    radius_m: float | None
    place: str | None        # "札幌站" etc.
    # If you already have lat/lon:
    lat: float | None = None
    lon: float | None = None

def time_to_slot(hhmm: str) -> int:
    h, m = hhmm.split(":")
    minutes = int(h)*60 + int(m)
    return int(round(minutes / SLOT_MIN)) % SLOTS_PER_DAY

def parse_query_rules(text: str, default_city="sapporo") -> CrowdQuery:
    # weekday
    wd = None
    for k,v in WEEKDAY_MAP.items():
        if k in text:
            wd = v
            break

    # time: "20:30"
    m = re.search(r"(\d{1,2}):(\d{2})", text)
    hhmm = None
    if m:
        hhmm = f"{int(m.group(1)):02d}:{int(m.group(2)):02d}"
    else:
        # crude: "晚上8點" / "8點"
        m2 = re.search(r"(晚上|夜晚|下午)?\s*(\d{1,2})\s*點(半)?", text)
        if m2:
            h = int(m2.group(2))
            if m2.group(1) in ("晚上","夜晚") and h < 12:
                h += 12
            if m2.group(1) == "下午" and h < 12:
                h += 12
            minute = 30 if m2.group(3) else 0
            hhmm = f"{h:02d}:{minute:02d}"

    # radius: "附近" -> default; "800m" -> parse
    radius_m = None
    m3 = re.search(r"(\d+(?:\.\d+)?)\s*(m|公尺|米|km|公里)", text.lower())
    if m3:
        val = float(m3.group(1))
        unit = m3.group(2)
        if unit in ("km","公里"):
            val *= 1000
        radius_m = val
    else:
        if "附近" in text:
            radius_m = 800  # default

    # place: very naive (you can do better: NER, dictionary, etc.)
    # Here: take first token before weekday/time keywords
    place = text
    for k in list(WEEKDAY_MAP.keys()) + ["附近"]:
        place = place.replace(k, " ")
    place = place.strip()
    place = place if place else None

    return CrowdQuery(city=default_city, weekday=wd, hhmm=hhmm, radius_m=radius_m, place=place)

In [ ]:
from openai import OpenAI
import json
import pandas as pd
from dotenv import load_dotenv

load_dotenv()      # 讀 .env 進環境變數
client = OpenAI()

QUERY_SCHEMA = {
  "name": "crowd_query",
  "schema": {
    "type": "object",
    "additionalProperties": False,
    "properties": {
      "city": {"type": "string"},
      "weekday": {"type": ["integer","null"], "minimum": 0, "maximum": 6},
      "hhmm": {"type": ["string","null"], "pattern": r"^\d{2}:\d{2}$"},
      "radius_m": {"type": ["number","null"], "minimum": 0},
      "place": {"type": ["string","null"]}
    },
    "required": ["city","weekday","hhmm","radius_m","place"]
  }
}

def parse_query_llm(text: str, default_city="sapporo") -> CrowdQuery:
    prompt = f"""
              你是人流查詢解析器。請把使用者輸入轉成 JSON（符合 schema）。
              - weekday: 0=週一 ... 6=週日
              - hhmm: 24小時制 "HH:MM"
              - radius_m: 公尺。若未指定且出現「附近」，填 800；否則 null。
              - place: 地點名稱字串，若沒有則 null
              - city 若沒提到，填 "{default_city}"

              使用者輸入：{text}
              """

    resp = client.responses.create(
        model="gpt-4o",  # 你可換成你要的模型
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": QUERY_SCHEMA["name"],
                "schema": QUERY_SCHEMA["schema"],
                "strict": True
            }
        }
    )
    data = json.loads(resp.output_text)
    return CrowdQuery(**data)

In [ ]:
import numpy as np
import sys
from src.util import *

def parse_grid_in_text(text: str):
    m = re.search(r"grid\((\d+)\s*,\s*(\d+)\)", text)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))

def answer_query(text: str, df_pred: pd.DataFrame, coverages=(0.5,0.8,0.95)):
    # 1) parse (先用 rule-based；你要換 LLM 就換這行)
    q = parse_query_rules(text, default_city="sapporo")

    if q.weekday is None or q.hhmm is None:
        return {"error": "缺少星期或時間，請補上例如：週六 20:00"}

    t_slot = time_to_slot(q.hhmm)

    # 2) resolve location -> grid center
    xy = parse_grid_in_text(text)
    if xy is None:
        # placeholder: 沒 mapping 就要求 grid
        return {"error": "目前尚未接格網地理映射，請先用 grid(x,y) 指定位置，例如：grid(120,50) 週六 20:00 附近"}
    x0, y0 = xy

    # 3) pick a representative day d for that weekday
    #    （正式版：你會用使用者指定日期；或用 “最近一個符合 weekday 的 d”）
    #    這裡做 demo：取 df_pred 裡 test 期間，第一個符合 weekday 的 day
    #    你也可以改成：讓 LLM 解析到具體日期。
    tmp_days = df_pred[["d"]].drop_duplicates().copy()
    tmp_days["weekday"] = (tmp_days["d"] % 7)  # 若你用真實日期weekday就改成你那套
    cand_days = tmp_days[tmp_days["weekday"] == q.weekday]["d"].tolist()
    if not cand_days:
        return {"error": "找不到對應 weekday 的資料日 d，請確認 weekday 定義與 d->weekday 映射一致"}
    d_use = int(cand_days[-1])  # 用最後一個（偏近期）

    # 4) get predicted slice at (d_use, t_slot)
    pred_slice = df_pred[(df_pred["d"]==d_use) & (df_pred["t"]==t_slot)][["x","y","score"]].copy()
    if pred_slice.empty:
        return {"error": "該時段沒有預測資料"}

    # 5) create circles around (x0,y0) within a neighborhood
    #    radius_cells 可由 radius_m 換算：radius_cells = radius_m / cell_size_m
    MAX_RADIUS_CELLS = 64
    radius_cells = 8 if q.radius_m is None else 8

    cells_xy = pred_slice[["x","y"]].to_numpy()
    dist = np.hypot(cells_xy[:,0]-x0, cells_xy[:,1]-y0)

    # 候選格：先取中心附近一個窗口，避免全圖計算太大
    cand = pred_slice[dist <= radius_cells + 1e-9]
    if len(cand) < 5:
        return {"error": "附近候選格太少，請加大 radius 或檢查格網"}

    cand_xy = cand[["x","y"]].to_numpy()
    p = normalize_nonneg(cand["score"].to_numpy())

    circles = []
    for alpha in coverages:
        win = radius_cells

        while True:
            cand = pred_slice[dist <= win + 1e-9]
            if len(cand) < 5:
                return {"error": "附近候選格太少，請加大 radius 或檢查格網"}

            cand_xy = cand[["x", "y"]].to_numpy()
            p = normalize_nonneg(cand["score"].to_numpy())

            # 固定圓心 (x0,y0)，找最小半徑讓圓內累積機率 >= alpha
            r, idx_circle = circle_radius_by_mass(cand_xy, p, x0, y0, alpha)
            achieved = float(p[idx_circle].sum())  # 實際達到的累積質量（可能 < alpha，若候選窗口太小）

            if achieved >= alpha or win >= MAX_RADIUS_CELLS:
                break

            win *= 2 

        
        circles.append({
            "alpha": alpha,
            "center_grid": {"x": float(x0), "y": float(y0)},  # 固定中心點
            "radius_cells": r,
            "n_cells": int(len(idx_circle)),
            "achieved_mass": achieved,
            "window_cells": win
        })

    return {
        "query": {
            "weekday": q.weekday, "hhmm": q.hhmm, "t_slot": t_slot,
            "d_used": d_use,
            "center_grid": {"x": x0, "y": y0},
            "radius_cells": radius_cells,
        },
        "circles": circles
    }

In [15]:
from src.geo.grid_to_latlng import GridLatLngMapper

anchors = [
    {"x": 24, "y": 151, "lat": 43.06918333153887, "lng": 141.35147072116592},  # 札幌站 
    {"x": 24, "y": 148, "lat": 43.07940372979633, "lng": 141.34225589803765},  # 北海道大學
    {"x": 26, "y": 153, "lat": 43.05798589528942, "lng": 141.35402112326315},  # 狸小路商店街
]

mapper = GridLatLngMapper(anchors)

In [16]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return float(2*R*np.arcsin(np.sqrt(a)))

def cell_size_m_at(mapper, x0, y0):
    pts = pd.DataFrame([{"x":x0,"y":y0},{"x":x0+1,"y":y0},{"x":x0,"y":y0+1}])
    pts = mapper.transform(pts)
    c = pts.iloc[0]
    px = pts.iloc[1]
    py = pts.iloc[2]
    dx = haversine_m(c.lat, c.lng, px.lat, px.lng)
    dy = haversine_m(c.lat, c.lng, py.lat, py.lng)
    return (dx + dy) / 2

In [35]:
import folium
from branca.colormap import linear

def plot_query_on_map(resp, mapper):
    x0 = resp["query"]["center_grid"]["x"]
    y0 = resp["query"]["center_grid"]["y"]
    d  = resp["query"]["d_used"]
    t  = resp["query"]["t_slot"]

    # grid to lat,lng
    center = mapper.transform(pd.DataFrame([{"x":x0,"y":y0}])).iloc[0]
    lat0, lng0 = float(center.lat), float(center.lng)

    cell_m = cell_size_m_at(mapper, x0, y0)

    # 取 95% 那個圈的 window
    c95 = next((c for c in resp["circles"] if abs(c["alpha"] - 0.95) < 1e-9), None)
    win = int(c95.get("window_cells", resp["query"]["radius_cells"])) if c95 else int(resp["query"]["radius_cells"])

    # 取 (d,t) 的預測切片，並限制在窗口內
    sl = df_pred[(df_pred["d"] == d) & (df_pred["t"] == t)][["x","y","score"]].copy()
    dist_grid = np.hypot(sl["x"] - x0, sl["y"] - y0)
    sl = sl[dist_grid <= win + 1e-9].copy()
    if sl.empty:
        raise ValueError("pred_slice is empty for this (d,t).")

    cand_xy = sl[["x","y"]].to_numpy()
    p_pred = normalize_nonneg(sl["score"].to_numpy())

    # 重新用同一批 cand_xy 算 95% 圈（固定中心）
    r95, idx95 = circle_radius_by_mass(cand_xy, p_pred, x0, y0, 0.95)
    achieved95 = float(p_pred[idx95].sum())

    m = folium.Map(location=[lat0, lng0], zoom_start=14)

    # 中心點 marker
    folium.Marker([lat0, lng0], tooltip=f"center grid=({x0},{y0})").add_to(m)

    # 圈圈
    for c in resp["circles"]:

        color = {
            0.5: "red",
            0.8: "orange",
            0.95: "blue"
        }[c["alpha"]]

        radius_m = float(c["radius_cells"] * cell_m)
        folium.Circle(
            location=[lat0, lng0],
            radius=radius_m,
            color=color,
            fill=True,
            fill_opacity=0.10,
            popup=f"alpha={c['alpha']}  r_cells={c['radius_cells']:.2f}  r_m≈{radius_m:.0f}"
        ).add_to(m)

    # 顏色映射（score 越大越紅）
    smin, smax = float(sl["score"].min()), float(sl["score"].max())
    cmap = linear.YlOrRd_09.scale(smin, smax)

    # 把每個格子轉成 lat/lng，並畫密度點；95% 圈內點加粗/更不透明
    ll = mapper.transform(sl[["x","y"]].copy())  # 需回傳含 lat/lng 欄位（依你的 mapper 實作）
    ll = ll.assign(score=sl["score"].values)

    # 計算每點是否在 95% 圈內（grid 距離 <= r95）
    dist_to_center = np.hypot(sl["x"].to_numpy() - x0, sl["y"].to_numpy() - y0)
    in95 = dist_to_center <= r95 + 1e-9

    for i in range(len(ll)):
        lat, lng = float(ll.iloc[i].lat), float(ll.iloc[i].lng)
        score = float(ll.iloc[i].score)
        color = cmap(score)

        # 點大小
        rr = 2 + 6 * (score - smin) / (smax - smin + 1e-12)

        folium.CircleMarker(
            location=[lat, lng],
            radius=float(rr),
            color="#000000" if in95[i] else "#666666",
            weight=2 if in95[i] else 1,
            fill=True,
            fill_color=color,
            fill_opacity=0.9 if in95[i] else 0.25,
            popup=f"score={score:.3f}  in95={bool(in95[i])}"
        ).add_to(m)


    # 顏色圖例
    cmap.caption = "predicted score"
    cmap.add_to(m)

    m.save("prdict_confidence.html")

In [43]:
query = answer_query("grid(20, 150) 週六 07:00 附近", df_pred)
plot_query_on_map(query, mapper)